# Best predictors based on calibration, permutation

## 0. Package loading and installation

In [1]:
# For Jupyter/Colab notebooks
%reset -f
import gc
gc.collect()

import numpy as np
import pandas as pd
from rpy2.robjects import r, pandas2ri

!pip install scikit-survival # Install scikit-survival if not already installed
!pip install miceforest --no-cache-dir
!pip install numpy cryptography

from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    integrated_brier_score
)
from sksurv.util import Surv

pandas2ri.activate()

#Glimpse function
def glimpse(df, max_width=80):
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].astype(str).head(5).tolist()
        preview_str = ", ".join(preview)
        if len(preview_str) > max_width:
            preview_str = preview_str[:max_width] + "..."
        print(f"{col:<30} {str(dtype):<15} {preview_str}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 11.6 MB/s eta 0:00:00


## Format

In [ ]:
from google.colab import userdata

# Names of the objects (and secrets)
object_names = [
    "times_eval_death",
    "times_eval_readm",
    "imputation_1"
]

for name in object_names:
    file_id = userdata.get(name)   # secret stored with this key
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")

    output_file = f"{name}.parquet"
    print(f"Downloading {name} -> {output_file}")
    !gdown --id {file_id} --output {output_file} --quiet

  # Names of the objects (and secrets)
object_names = [
    "X_reduced_list.npz.gpg"
]

for name in object_names:
    file_id = userdata.get(name)   # secret stored with this key
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")

    output_file = f"{name}"
    print(f"Downloading {name} -> {output_file}")
    !gdown --id {file_id} --output {output_file} --quiet

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(


In [ ]:
from google.colab import userdata
import subprocess
import os
import sys

# 1. Create dedicated writable directory
output_path = '/content/X_reduced_list.npz'

# 2. Get passphrase with special character handling
passphrase = userdata.get('passphrase')
if not passphrase:
    raise ValueError("❌ Passphrase not set in Colab userdata. Set via: userdata.set('passphrase', 'your_pass')")

# 3. Decrypt with explicit binary handling
try:
    result = subprocess.run([
        'gpg', '--batch',
        '--yes',  # Overwrite files without confirmation
        '--pinentry-mode', 'loopback',
        '--passphrase', passphrase,
        '--output', output_path,
        '--decrypt', 'X_reduced_list.npz.gpg'
    ],
    capture_output=True,
    text=True,
    check=True  # Raise exception on failure
    )

    print(f"✅ SUCCESS: Decrypted file saved to:\n{output_path}")
    print(f"📦 File size: {os.path.getsize(output_path) / 1024:.1f} KB")

except subprocess.CalledProcessError as e:
    print(f"❌ GPG ERROR (exit code {e.returncode}):")
    stderr = e.stderr.strip()

    # Special handling for common errors
    if "Bad session key" in stderr:
        print("🔑 PASSPHRASE MISMATCH: Verify your passphrase in Colab userdata")
        print("   → Set correct passphrase with: userdata.set('passphrase', 'CORRECT_PASS')")
    elif "No such file" in stderr:
        print("📂 MISSING ENCRYPTED FILE: Upload X_reduced_list.npz.gpg to Colab first")
        print("   → Use left sidebar '📁 Files' tab to upload")
    else:
        print(f"📄 Raw GPG error:\n{stderr}")

    # Show debug info
    print("\n🔍 DEBUG INFO:")
    print(f"- Current directory: {os.getcwd()}")
    print(f"- Files in directory: {os.listdir()}")
    print(f"- Passphrase length: {len(passphrase)} characters (hidden for security)")
    sys.exit(1)

✅ SUCCESS: Decrypted file saved to:
/content/X_reduced_list.npz
📦 File size: 23437.3 KB


Load objects in Python

In [ ]:
# ---- Time grids ----
times_eval_death = pd.read_parquet("times_eval_death.parquet")["time"].to_numpy()
times_eval_readm = pd.read_parquet("times_eval_readm.parquet")["time"].to_numpy()

print("times_eval_death shape:", times_eval_death.shape)
print("times_eval_readm shape:", times_eval_readm.shape)


times_eval_death shape: (49,)
times_eval_readm shape: (50,)


Added outcomes

In [ ]:
import pandas as pd
import numpy as np
from google.colab import userdata

# ---- Download y_surv_readm & y_surv_death ----
for name in ["y_surv_readm", "y_surv_death"]:
    file_id = userdata.get(name)
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")
    !gdown --id {file_id} --output {name}.parquet --quiet
    print(f"Downloaded {name}.parquet")

# ---- Load and reconstruct Surv-like structured arrays ----
df_y_readm = pd.read_parquet("y_surv_readm.parquet")
y_surv_readm = np.array(
    list(zip(df_y_readm["event"].astype(bool),
             df_y_readm["time"].astype(float))),
    dtype=[("event", "?"), ("time", "<f8")]
)

df_y_death = pd.read_parquet("y_surv_death.parquet")
y_surv_death = np.array(
    list(zip(df_y_death["event"].astype(bool),
             df_y_death["time"].astype(float))),
    dtype=[("event", "?"), ("time", "<f8")]
)

print("y_surv_readm shape:", y_surv_readm.shape)
print("y_surv_death shape:", y_surv_death.shape)


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloaded y_surv_readm.parquet
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloaded y_surv_death.parquet
y_surv_readm shape: (88504,)
y_surv_death shape: (88504,)


In [ ]:
import numpy as np
import pandas as pd

# Load the data
data = np.load("X_reduced_list.npz", allow_pickle=True)

imputation_1 = pd.read_parquet("imputation_1.parquet")

# Reconstruct as DataFrames (you'll need the original column names)
# Your full original feature list (80 features)
all_features = [
    'readmit_time_from_adm_m',
    'death_time_from_adm_m',
    'adm_age_rec3',
    'porc_pobr',
    'dit_m',
    'national_foreign',
    'ethnicity',
    'dg_psiq_cie_10_instudy',
    'dg_psiq_cie_10_dg',
    'dx_f3_mood',
    'dx_f6_personality',
    'dx_f_any_severe_mental',
    'any_phys_dx',
    'polysubstance_strict',
    'readmit_time_from_disch_m',      # ← outcome to exclude
    'readmit_event',                  # ← outcome to exclude
    'death_time_from_disch_m',        # ← outcome to exclude
    'death_event',                    # ← outcome to exclude
    'sex_rec_woman',
    'tenure_status_household_illegal_settlement',
    'tenure_status_household_owner_transferred_dwellings_pays_dividends',
    'tenure_status_household_renting',
    'tenure_status_household_stays_temporarily_with_a_relative',
    'cohabitation_alone',
    'cohabitation_with_couple_children',
    'cohabitation_family_of_origin',
    'sub_dep_icd10_status_drug_dependence',
    'any_violence_1_domestic_violence_sex_abuse',
    'prim_sub_freq_rec_2_2_6_days_wk',
    'prim_sub_freq_rec_3_daily',
    'tr_outcome_adm_discharge_adm_reasons',
    'tr_outcome_adm_discharge_rule_violation_undet',
    'tr_outcome_completion',
    'tr_outcome_dropout',
    'tr_outcome_referral',
    'adm_motive_another_sud_facility_fonodrogas_senda_previene',
    'adm_motive_justice_sector',
    'adm_motive_sanitary_sector',
    'adm_motive_spontaneous_consultation',
    'first_sub_used_alcohol',
    'first_sub_used_cocaine_paste',
    'first_sub_used_cocaine_powder',
    'first_sub_used_marijuana',
    'first_sub_used_opioids',
    'first_sub_used_tranquilizers_hypnotics',
    'primary_sub_mod_cocaine_paste',
    'primary_sub_mod_cocaine_powder',
    'primary_sub_mod_alcohol',
    'primary_sub_mod_marijuana',
    'tipo_de_vivienda_rec2_other_unknown',
    'plan_type_corr_m_pai',
    'plan_type_corr_m_pr',
    'plan_type_corr_pg_pai',
    'plan_type_corr_pg_pr',
    'occupation_condition_corr24_inactive',
    'occupation_condition_corr24_unemployed',
    'marital_status_rec_separated_divorced_annulled_widowed',
    'marital_status_rec_single',
    'urbanicity_cat_1_rural',
    'urbanicity_cat_2_mixed',
    'ed_attainment_corr_2_completed_high_school_or_less',
    'ed_attainment_corr_3_completed_primary_school_or_less',
    'evaluacindelprocesoteraputico_logro_intermedio',
    'evaluacindelprocesoteraputico_logro_minimo',
    'eva_consumo_logro_intermedio',
    'eva_consumo_logro_minimo',
    'eva_fam_logro_intermedio',
    'eva_fam_logro_minimo',
    'eva_relinterp_logro_intermedio',
    'eva_relinterp_logro_minimo',
    'eva_ocupacion_logro_intermedio',
    'eva_ocupacion_logro_minimo',
    'eva_sm_logro_intermedio',
    'eva_sm_logro_minimo',
    'eva_fisica_logro_intermedio',
    'eva_fisica_logro_minimo',
    'eva_transgnorma_logro_intermedio',
    'eva_transgnorma_logro_minimo',
    'id_centro'
]

outcome_indices = [
    all_features.index('readmit_time_from_disch_m'),
    all_features.index('readmit_time_from_adm_m'),
    all_features.index('readmit_event'),
    all_features.index('death_time_from_disch_m'),
    all_features.index('death_time_from_adm_m'),
    all_features.index('death_event'),
    all_features.index('id_centro')
]

# Combine all indices to remove
cols_to_remove = sorted(set(outcome_indices))

# Create new feature list WITHOUT these columns
feature_columns = [
    feat for i, feat in enumerate(all_features)
    if i not in cols_to_remove
]

# Now build X_reduced_list using ONLY the kept columns
X_reduced_list = [
    pd.DataFrame(data['X_reduced_list_1'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_2'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_3'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_4'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_5'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns)
]

# 3. “Best predictors” (variable importance) based on calibration

To identify predictors that most influenced overall prediction error (calibration and discrimination combined), we computed permutation importance using the Integrated Brier Score (IBS). Within each imputed dataset, we ran k-fold cross-validation and fitted Coxnet models in the training folds. For each fold, we first obtained the out-of-sample IBS using the test data. We then permuted one predictor at a time in the test set, recomputed survival predictions and the IBS, and recorded the increase in IBS (worsening of prediction error). These IBS increases were pooled across folds and imputations, so that `mean_increase_ibs` summarized how much each predictor worsened out-of-sample IBS on average, while respecting both multiple imputation and cross-validation. Sorting predictors by `mean_increase_ibs` and selecting the top 20 yielded the most influential predictors from the standpoint of overall calibration-weighted predictive accuracy.

It fits a Coxnet model (a penalized Cox proportional hazards model from the sksurv library) and measures how much each feature impacts the model's performance, specifically using the Integrated Brier Score (IBS) as the metric. IBS is a measure of prediction error in survival models—lower is better, so permuting (randomly shuffling) a feature and seeing how much IBS increases indicates that feature's importance.

The function runs CV across folds and imputations, fits the model, computes a baseline IBS, then repeatedly permutes each feature's values in the test set to see the "drop" (increase) in IBS. It aggregates these across all runs to rank features by their mean IBS increase (higher increase = more important feature).

First, we eliminated inmortal time bias (dead patients look like without readmission).

This correction is essentially the Cause-Specific Hazard preparation. It is the correct way to handle Aim 3 unless you switch to a Fine-Gray model (which treats death as a specific type of event 2, rather than censoring 0). For RSF/Coxnet, censoring 0 is the correct approach.

In [ ]:
import numpy as np
import pandas as pd

def correct_competing_risks(X_list, y_readm_list, y_death_list):
    """
    Corrects Readmission time/event using Death data (Competing Risk).
    If Death happens BEFORE Readmission, we censor Readmission at the time of Death.
    """
    new_y_readm_list = []

    print("Correcting Readmission outcomes for Competing Risk (Death)...")

    for i, (y_r, y_d) in enumerate(zip(y_readm_list, y_death_list)):
        # Extract arrays
        r_time = y_r["time"]
        r_event = y_r["event"]

        d_time = y_d["time"]
        d_event = y_d["event"]

        # Create copies to modify
        new_r_time = r_time.copy()
        new_r_event = r_event.copy()

        # LOGIC:
        # Find cases where Death happened (d_event=1) AND Death was earlier than Readmission Time
        # Note: We compare times. If death is earlier, the patient never reached the "readmission" time potentially recorded.

        # Mask: Patients who died BEFORE the recorded readmission/censoring time
        # (or at the same time, if they died without being readmitted)
        mask_death_first = (d_event == True) & (d_time <= r_time)

        count_corrections = np.sum(mask_death_first)

        # 1. Update Time: Cut off at death time
        new_r_time[mask_death_first] = d_time[mask_death_first]

        # 2. Update Event: Force to 0 (Censored) because they died, not readmitted
        # (Unless they were readmitted exactly at the moment of death, which is unlikely/impossible in this context)
        new_r_event[mask_death_first] = False

        # Reconstruct structured array
        new_y_surv = np.empty(len(new_r_time), dtype=[("event", "?"), ("time", "<f8")])
        new_y_surv["event"] = new_r_event
        new_y_surv["time"] = new_r_time

        new_y_readm_list.append(new_y_surv)

        print(f"  Imputation {i+1}: Corrected {count_corrections} patients who died before readmission end-point.")

    return new_y_readm_list

# ==============================================================================
# EXECUTION
# ==============================================================================

# Create lists of outcomes (repeating the same observed outcome for each imputation)
# This is necessary because the function iterates over (X, y) pairs.
n_imputations = len(X_reduced_list)
y_surv_readm_list = [y_surv_readm for _ in range(n_imputations)]
y_surv_death_list = [y_surv_death for _ in range(n_imputations)]

# Run this correction BEFORE fitting your models
y_surv_readm_list_corrected = correct_competing_risks(
    X_reduced_list,
    y_surv_readm_list,
    y_surv_death_list
)

# Now use `y_surv_readm_list_corrected` for your Readmission RSF/Cox models.

Correcting Readmission outcomes for Competing Risk (Death)...
  Imputation 1: Corrected 3203 patients who died before readmission end-point.
  Imputation 2: Corrected 3203 patients who died before readmission end-point.
  Imputation 3: Corrected 3203 patients who died before readmission end-point.
  Imputation 4: Corrected 3203 patients who died before readmission end-point.
  Imputation 5: Corrected 3203 patients who died before readmission end-point.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import brier_score, integrated_brier_score
from joblib import Parallel, delayed

def permutation_importance_ibs_cv_mi2(
    X_list,
    y_surv_list, # Renamed to y_surv_list to reflect it's a list of outcomes
    times_eval,
    alpha_idx=25,
    n_splits=5,
    n_repeats=3,
    random_state=2125,
    l1_ratio=0.9,
    alpha_min_ratio=0.01,
    n_alphas=50,
    max_iter=100000,
    n_jobs=-1,
):
    """
    Highly optimized version using risk scores and coefficient-based updates.

    Parameters
    ----------
    X_list : list of pandas.DataFrame
        List of imputed datasets (same samples, different imputations)
    y_surv_list : list of structured array
        List of survival outcomes (time, event), one for each imputed dataset
    times_eval : array-like
        Time points for evaluation
    alpha_idx : int
        Index of alpha to use from regularization path
    n_splits : int
        Number of CV folds
    n_repeats : int
        Number of permutation repeats per feature
    random_state : int
        Random seed for reproducibility
    l1_ratio : float
        ElasticNet mixing parameter (0 <= l1_ratio <= 1)
    alpha_min_ratio : float
        Minimum alpha as fraction of maximum alpha
    n_alphas : int
        Number of alphas along regularization path
    max_iter : int
        Maximum iterations for model fitting
    n_jobs : int
        Number of parallel jobs (-1 for all cores)

    Returns
    -------
    baseline_ibs_mean : float
        Mean baseline IBS across folds and imputations
    baseline_ibs_sd : float
        Standard deviation of baseline IBS
    df_imp_ibs : pandas.DataFrame
        Dataframe with permutation importance results
    """
    # Convert to NumPy arrays upfront
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_list = [X.values.astype(np.float64, copy=False) for X in X_list]
    times_eval = np.asarray(times_eval, dtype=np.float64)
    n_imputations = len(X_list)

    # Precompute CV splits once
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = list(kf.split(np.arange(X_list[0].shape[0])))

    def compute_fold_optimized(d, fold_idx, train_idx, test_idx):
        """
        Optimized fold computation using coefficient-based risk updates.
        """
        X_imp = X_list[d]
        y_imp = y_surv_list[d] # Get the y for *this* imputation

        X_train = X_imp[train_idx]
        X_test = X_imp[test_idx]
        y_train = y_imp[train_idx] # Slice the y for this imputation
        y_test = y_imp[test_idx]   # Slice the y for this imputation

        local_rng = np.random.RandomState(random_state + d * n_splits + fold_idx)

        # Fit Coxnet
        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alpha_min_ratio=alpha_min_ratio,
            n_alphas=n_alphas,
            normalize=False,
            fit_baseline_model=True,
            max_iter=max_iter,
            verbose=False,
        )
        model.fit(X_train, y_train)

        # Handle alpha selection safely
        eff_alpha_idx = min(alpha_idx, len(model.alphas_) - 1)
        alpha_val = model.alphas_[eff_alpha_idx]

        # Extract coefficients for risk score computation
        coef = model.coef_[:, eff_alpha_idx]

        # Access baseline survival
        baseline_estimator = model._baseline_models[eff_alpha_idx]
        S0 = baseline_estimator.baseline_survival_(times_eval)

        # Precompute baseline risk scores
        baseline_risks = model.predict(X_test, alpha=alpha_val)

        # Compute baseline survival probabilities
        S_test = np.power(S0[None, :], np.exp(baseline_risks[:, None]))

        # Get effective time grid for Brier score
        times_brier, _ = brier_score(
            survival_train=y_train,
            survival_test=y_test,
            estimate=S_test,
            times=times_eval,
        )
        n_times = len(times_brier)
        S_test_valid = S_test[:, :n_times]

        # Baseline IBS
        ibs_base = integrated_brier_score(
            survival_train=y_train,
            survival_test=y_test,
            estimate=S_test_valid,
            times=times_brier,
        )

        # Optimized permutation importance using risk score updates
        fold_drops = np.zeros((n_features, n_repeats))

        for col_idx in range(n_features):
            col_original = X_test[:, col_idx].copy()
            coef_col = coef[col_idx]

            for r in range(n_repeats):
                # Generate permutation
                permuted_values = local_rng.permutation(col_original)

                # Fast risk score update instead of full prediction
                risk_delta = (permuted_values - col_original) * coef_col
                risks_perm = baseline_risks + risk_delta

                # Compute survival probabilities from updated risks
                S_perm = np.power(S0[None, :], np.exp(risks_perm[:, None]))
                S_perm_valid = S_perm[:, :n_times]

                # Compute IBS for permuted feature
                ibs_perm = integrated_brier_score(
                    survival_train=y_train,
                    survival_test=y_test,
                    estimate=S_perm_valid,
                    times=times_brier,
                )
                fold_drops[col_idx, r] = ibs_perm - ibs_base

        return ibs_base, fold_drops

    print(f"\n=== {n_imputations} imputations – {n_splits}-fold CV permutation importance (IBS) ===")
    print(f"Total iterations: {n_imputations * n_splits}")
    print(f"Features: {n_features}, Permutation repeats: {n_repeats}")

    # Parallel execution
    results = Parallel(n_jobs=n_jobs, backend='loky', verbose=10)(
        delayed(compute_fold_optimized)(d, fold_idx, train_idx, test_idx)
        for d in range(n_imputations)
        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits)
    )

    # Aggregate results
    baseline_ibs_list = [res[0] for res in results]

    # Concatenate all permutation results
    all_drops = np.concatenate([res[1] for res in results], axis=1)

    # Create results dataframe
    df_imp_ibs = pd.DataFrame({
        "feature": feature_names,
        "mean_increase_ibs": all_drops.mean(axis=1),
        "sd_increase_ibs": all_drops.std(axis=1, ddof=1),
        "n_evals": all_drops.shape[1],
    }).sort_values("mean_increase_ibs", ascending=False).reset_index(drop=True)

    # Baseline IBS statistics
    baseline_ibs_mean = np.mean(baseline_ibs_list)
    baseline_ibs_sd = np.std(baseline_ibs_list, ddof=1) if len(baseline_ibs_list) > 1 else 0.0

    print("\n=== Baseline CV IBS over imputations & folds ===")
    print(f"Mean \u00B1 SD: {baseline_ibs_mean:.4f} \u00B1 {baseline_ibs_sd:.4f}")
    print(f"Range: [{np.min(baseline_ibs_list):.4f}, {np.max(baseline_ibs_list):.4f}]")

    print("\n=== Top 20 Most Important Features ===")
    print(df_imp_ibs.head(20).to_string(index=False))

    return baseline_ibs_mean, baseline_ibs_sd, df_imp_ibs

### Execute
- **`X_list`** (set to `X_reduced_list` in your code): This is a list of Pandas DataFrames. The function loops over each imputation separately during CV, fitting models and computing importances for robustness.

- **`y_surv`** (set to `y_surv_readm`): This is your survival outcome/target variable, as a NumPy structured array with two fields: 'event' (bool: True if event like death/readmission occurred, False if censored) and 'time' (float: time to event or censoring).

- **`times_eval`** (set to `times_eval_readm`): This is a NumPy array or list of evenly spaced time points covering your data's range. More points = finer evaluation but slightly higher computation. Too few = coarse/less accurate IBS.

- **`alpha_idx`** (set to 25, default=25): This is the index in the Coxnet model's alpha path (regularization strengths) to use for predictions and importance. Selects the effective alpha for the model after fitting (e.g., index 0 is strongest regularization, higher indices are weaker). Typical values range 0-50. Lower index = more regularization (a sparser model with fewer features used); Higher = less regularization (more complex model). Tune via separate CV if needed.

- **`n_splits`** (set to 5, default=5):  This is the number of folds in K-Fold CV.  Splits data into train/test sets repeatedly (e.g., 5 folds = 80% train, 20% test per fold). Precomputed once and reused across imputations.  Balances bias-variance; 5 is a good default for moderate datasets.

- **`n_repeats`** (set to 5, default=3): This is the number of times to repeat the permutation for each feature in each fold/imputation. For each feature, shuffles its test-set values `n_repeats` times, recomputes IBS each time, and averages the drops for stability. Increase for better precision and more reliable importance scores (lower SD in output DF), but runtime increases linearly (e.g., doubles if going from 3 to 6).

### Default Arguments (Not Specified in Your Call, So They Use These Values)
These are less commonly changed but control the underlying Coxnet model and computation:
- `random_state=2125`: Seed for CV splitting and permutations (ensures reproducibility).
- `l1_ratio=0.9`: Elastic net mixing (0.9 = mostly L1/Lasso for feature selection, 0 = L2/Ridge).
- `alpha_min_ratio=0.01`: Smallest alpha as a fraction of the max (controls regularization range).
- `max_iter=100000`: Max iterations for model convergence (high to avoid early stopping).
- `n_jobs=-1`: Parallel jobs (uses all CPU cores for speed via Joblib).

In [ ]:
baseline_ibs_readm, baseline_ibs_sd_readm, df_imp_ibs_readm = (
    permutation_importance_ibs_cv_mi2(
        X_list=X_reduced_list,
        y_surv_list=y_surv_readm_list_corrected,
        times_eval=times_eval_readm,
        alpha_idx=25,   # same index you’ve been using
        n_splits=10,
        n_repeats=10,     # you can increase to 5 if runtime is acceptable
        # More flexible regularization:
        l1_ratio=0.5,             # ← BALANCED Lasso/Ridge (was 0.9): 50% Lasso + 50% Ridge keeps more features active
        alpha_min_ratio=0.1,      # ← LESS aggressive alpha range (was 0.01):  Focuses on less regularized models
        n_alphas=50,
        max_iter=100000,
        n_jobs=-1
    )
)
#[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed: 67.9min finished

# Top 20 predictors for readmission, ranked by increase in IBS
df_imp_ibs_readm.head(20)


=== 5 imputations – 10-fold CV permutation importance (IBS) ===
Total iterations: 50
Features: 72, Permutation repeats: 10


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:  4.1min
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:  4.1min
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:  8.1min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed: 10.2min
[Parallel(n_jobs=-1)]: Done  41 out of  50 | elapsed: 12.2min remaining:  2.7min
[Parallel(n_jobs=-1)]: Done  47 out of  50 | elapsed: 12.2min remaining:   46.7s



=== Baseline CV IBS over imputations & folds ===
Mean ± SD: 0.1441 ± 0.0016
Range: [0.1425, 0.1472]

=== Top 20 Most Important Features ===
                                                           feature  mean_increase_ibs  sd_increase_ibs  n_evals
                                   prim_sub_freq_rec_2_2_6_days_wk       6.748184e-08     5.965911e-07      500
tenure_status_household_owner_transferred_dwellings_pays_dividends       1.088289e-08     1.124657e-07      500
                               tipo_de_vivienda_rec2_other_unknown       1.299106e-09     8.175857e-08      500
                                                         porc_pobr       0.000000e+00     0.000000e+00      500
                                                 dg_psiq_cie_10_dg       0.000000e+00     0.000000e+00      500
                                                  national_foreign       0.000000e+00     0.000000e+00      500
                                            dg_psiq_cie_10_instudy       0.

[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed: 13.3min finished


,feature,mean_increase_ibs,sd_increase_ibs,n_evals
0,prim_sub_freq_rec_2_2_6_days_wk,6.748184e-08,5.965911e-07,500
1,tenure_status_household_owner_transferred_dwel...,1.088289e-08,1.124657e-07,500
2,tipo_de_vivienda_rec2_other_unknown,1.299106e-09,8.175857e-08,500
3,porc_pobr,0.000000e+00,0.000000e+00,500
4,dg_psiq_cie_10_dg,0.000000e+00,0.000000e+00,500
5,national_foreign,0.000000e+00,0.000000e+00,500
6,dg_psiq_cie_10_instudy,0.000000e+00,0.000000e+00,500
7,ethnicity,0.000000e+00,0.000000e+00,500
8,primary_sub_mod_cocaine_paste,0.000000e+00,0.000000e+00,500
9,primary_sub_mod_cocaine_powder,0.000000e+00,0.000000e+00,500


In [ ]:
baseline_ibs_death, baseline_ibs_sd_death, df_imp_ibs_death = (
    permutation_importance_ibs_cv_mi2(
        X_list=X_reduced_list,
        y_surv_list=y_surv_death_list, # Changed from y_surv=y_surv_death
        times_eval=times_eval_death,
        alpha_idx=25,
        n_splits=10,
        n_repeats=10,
        # More flexible regularization:
        l1_ratio=0.5,             # ← BALANCED Lasso/Ridge (was 0.9): 50% Lasso + 50% Ridge keeps more features active
        alpha_min_ratio=0.1,      # ← LESS aggressive alpha range (was 0.01):  Focuses on less regularized models
        n_alphas=50,
        max_iter=100000,
        n_jobs=-1
    )
)
# Top 20 predictors for death
df_imp_ibs_death.head(20)


=== 5 imputations – 10-fold CV permutation importance (IBS) ===
Total iterations: 50
Features: 72, Permutation repeats: 10


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


KeyboardInterrupt: 

# Landmark

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import brier_score, integrated_brier_score
from joblib import Parallel, delayed

def permutation_importance_calibration_timespecific_cv_mi(
    X_list,
    y_surv_list,
    times_eval,
    alpha_idx=25,
    n_splits=5,
    n_repeats=3,
    random_state=2125,
    l1_ratio=0.1,
    alpha_min_ratio=0.001,
    n_alphas=50,
    max_iter=100000,
    n_jobs=-1,
):
    """
    Multiple-imputation + k-fold CV permutation importance for Coxnet
    with time-specific CALIBRATION metrics (Brier Score).

    Parameters
    ----------
    X_list : list of pandas.DataFrame
        List of imputed design matrices (one per imputation).
    y_surv_list : list of structured arrays
        List of Surv(event, time) structured arrays (one per imputation).
    times_eval : array-like
        Time points at which to evaluate calibration.
    alpha_idx : int
        Index along the Coxnet regularization path to use for predictions.
    n_splits : int
        Number of CV folds.
    n_repeats : int
        Number of permutations per feature per fold.
    random_state : int
        Seed for KFold and permutations.
    l1_ratio : float
        ElasticNet mixing parameter.
    alpha_min_ratio : float
        Minimum alpha as fraction of maximum alpha.
    n_alphas : int
        Number of alphas along regularization path.
    max_iter : int
        Maximum iterations for model fitting.
    n_jobs : int
        Number of parallel jobs (-1 = all cores).

    Returns
    -------
    df_global_metrics : pandas.DataFrame
        Global calibration metrics (IBS).
    df_time_metrics : pandas.DataFrame
        Time-specific calibration metrics (BS(t)) for each time point.
    df_importance : pandas.DataFrame
        Feature-level permutation importance (IBS-based).
    """
    # Convert to NumPy arrays upfront for speed
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_list = [X.values.astype(np.float64, copy=False) for X in X_list]
    times_eval = np.asarray(times_eval, dtype=np.float64)
    n_imputations = len(X_list)

    # Filter times_eval to be within data range
    max_time = np.max([y['time'].max() for y in y_surv_list])
    times_eval = times_eval[times_eval <= max_time]
    n_times = len(times_eval)

    # Precompute CV splits once
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = list(kf.split(np.arange(X_list[0].shape[0])))

    def format_time_label(months):
        """Convert months to readable label."""
        if months < 1:
            return f"{int(months*4)}wk"
        elif months < 12:
            return f"{int(months)}mo"
        else:
            return f"{int(months/12)}yr"

    def compute_fold_calibration(d, fold_idx, train_idx, test_idx):
        """
        Compute calibration metrics for one imputation-fold.

        Returns:
            - baseline_ibs: Integrated Brier Score
            - bs_at_times: Brier Score at each time point
            - fold_drops: Permutation importance (IBS increase)
        """
        print(f"  Imputation {d+1}/{n_imputations}, fold {fold_idx+1}/{n_splits}")

        X_imp = X_list[d]
        y_imp = y_surv_list[d]

        X_train = X_imp[train_idx]
        X_test = X_imp[test_idx]
        y_train = y_imp[train_idx]
        y_test = y_imp[test_idx]

        local_rng = np.random.RandomState(random_state + d * n_splits + fold_idx)

        # Fit Coxnet
        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alpha_min_ratio=alpha_min_ratio,
            n_alphas=n_alphas,
            normalize=False,
            fit_baseline_model=True,
            max_iter=max_iter,
            verbose=False,
        )
        model.fit(X_train, y_train)

        # Handle alpha selection safely
        eff_alpha_idx = min(alpha_idx, len(model.alphas_) - 1)
        alpha_val = model.alphas_[eff_alpha_idx]

        # Extract coefficients for risk score computation
        coef = model.coef_[:, eff_alpha_idx]

        # Access baseline survival
        baseline_estimator = model._baseline_models[eff_alpha_idx]
        S0 = baseline_estimator.baseline_survival_(times_eval)

        # Precompute baseline risk scores
        baseline_risks = model.predict(X_test, alpha=alpha_val)

        # Compute baseline survival probabilities
        S_test_baseline = np.power(S0[None, :], np.exp(baseline_risks[:, None]))

        # 1. Compute time-specific Brier Scores BS(t)
        bs_at_times = []
        for t_idx, t in enumerate(times_eval):
            try:
                # Brier score at specific time t
                times_grid, bs_values = brier_score(
                    survival_train=y_train,
                    survival_test=y_test,
                    estimate=S_test_baseline,
                    times=times_eval,
                )
                # Find the BS value at time t
                bs_at_t = bs_values[t_idx]
                bs_at_times.append(bs_at_t)
            except Exception as e:
                print(f"    Warning: BS calculation failed at t={t} - {str(e)}")
                bs_at_times.append(np.nan)

        bs_at_times = np.array(bs_at_times)

        # 2. Compute Integrated Brier Score (IBS)
        try:
            baseline_ibs = integrated_brier_score(
                survival_train=y_train,
                survival_test=y_test,
                estimate=S_test_baseline,
                times=times_eval,
            )
        except Exception as e:
            print(f"    Warning: IBS calculation failed - {str(e)}")
            baseline_ibs = np.nan

        # 3. Optimized permutation importance using risk score updates
        fold_drops = np.zeros((n_features, n_repeats))

        for col_idx in range(n_features):
            col_original = X_test[:, col_idx].copy()
            coef_col = coef[col_idx]

            for r in range(n_repeats):
                # Generate permutation
                permuted_values = local_rng.permutation(col_original)

                # Fast risk score update instead of full prediction
                risk_delta = (permuted_values - col_original) * coef_col
                risks_perm = baseline_risks + risk_delta

                # Compute survival probabilities from updated risks
                S_perm = np.power(S0[None, :], np.exp(risks_perm[:, None]))

                # Compute IBS for permuted feature
                try:
                    ibs_perm = integrated_brier_score(
                        survival_train=y_train,
                        survival_test=y_test,
                        estimate=S_perm,
                        times=times_eval,
                    )
                    fold_drops[col_idx, r] = ibs_perm - baseline_ibs
                except:
                    fold_drops[col_idx, r] = 0.0

        return {
            'ibs': baseline_ibs,
            'bs_at_times': bs_at_times,
            'fold_drops': fold_drops
        }

    print(f"\n{'='*80}")
    print(f"TIME-SPECIFIC CALIBRATION EVALUATION (BRIER SCORE)")
    print(f"{'='*80}")
    print(f"Imputations: {n_imputations}, CV folds: {n_splits}")
    print(f"Time points: {len(times_eval)}")
    print(f"Times (months): {times_eval}")
    print(f"{'='*80}\n")

    # Parallelize over all imputation-fold combinations
    results = Parallel(n_jobs=n_jobs, backend='loky', verbose=10)(
        delayed(compute_fold_calibration)(d, fold_idx, train_idx, test_idx)
        for d in range(n_imputations)
        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits)
    )

    # ==================== COLLECT RESULTS ====================

    # 1. Collect IBS values
    baseline_ibs_list = [res['ibs'] for res in results if not np.isnan(res['ibs'])]

    # 2. Collect time-specific BS(t) values
    all_bs_at_times = np.array([res['bs_at_times'] for res in results])  # shape: (n_folds, n_times)

    # 3. Collect permutation importance
    all_drops = np.concatenate([res['fold_drops'] for res in results], axis=1)

    # ==================== AGGREGATE GLOBAL METRICS ====================
    ibs_mean = np.mean(baseline_ibs_list)
    ibs_sd = np.std(baseline_ibs_list, ddof=1) if len(baseline_ibs_list) > 1 else 0.0

    df_global = pd.DataFrame([{
        'ibs_mean': ibs_mean,
        'ibs_sd': ibs_sd,
        'ibs_min': np.min(baseline_ibs_list),
        'ibs_max': np.max(baseline_ibs_list),
        'n_folds_total': len(baseline_ibs_list),
        'n_timepoints': n_times
    }])

    # ==================== AGGREGATE TIME-SPECIFIC METRICS ====================
    time_rows = []
    for i, t in enumerate(times_eval):
        bs_at_t = all_bs_at_times[:, i]
        bs_at_t_valid = bs_at_t[~np.isnan(bs_at_t)]

        if len(bs_at_t_valid) > 0:
            time_rows.append({
                'time_months': t,
                'time_label': format_time_label(t),
                'bs_mean': np.mean(bs_at_t_valid),
                'bs_sd': np.std(bs_at_t_valid, ddof=1) if len(bs_at_t_valid) > 1 else 0.0,
                'bs_min': np.min(bs_at_t_valid),
                'bs_max': np.max(bs_at_t_valid),
                'n_evals': len(bs_at_t_valid)
            })
        else:
            time_rows.append({
                'time_months': t,
                'time_label': format_time_label(t),
                'bs_mean': np.nan,
                'bs_sd': np.nan,
                'bs_min': np.nan,
                'bs_max': np.nan,
                'n_evals': 0
            })

    df_time = pd.DataFrame(time_rows)

    # ==================== AGGREGATE FEATURE IMPORTANCE ====================
    df_importance = pd.DataFrame({
        "feature": feature_names,
        "mean_increase_ibs": all_drops.mean(axis=1),
        "sd_increase_ibs": all_drops.std(axis=1, ddof=1),
        "n_evals": all_drops.shape[1],
    }).sort_values("mean_increase_ibs", ascending=False).reset_index(drop=True)

    # ==================== PRINT SUMMARY ====================
    print(f"\n{'='*80}")
    print(f"CALIBRATION RESULTS SUMMARY")
    print(f"{'='*80}")
    print(f"\n>>> GLOBAL CALIBRATION METRICS <<<")
    print(f"IBS (Integrated Brier Score):  {ibs_mean:.4f} ± {ibs_sd:.4f}")
    print(f"IBS Range: [{np.min(baseline_ibs_list):.4f}, {np.max(baseline_ibs_list):.4f}]")

    print(f"\n>>> TIME-SPECIFIC CALIBRATION (Brier Score at each time) <<<")
    print(df_time[['time_label', 'bs_mean', 'bs_sd', 'n_evals']].to_string(index=False))

    print(f"\n>>> TOP 10 FEATURES (by IBS increase when permuted) <<<")
    print(df_importance.head(10).to_string(index=False))
    print(f"{'='*80}\n")

    return df_global, df_time, df_importance


# ==================== VISUALIZATION ====================
def plot_time_dependent_calibration(df_time, outcome_name="Readmission"):
    """
    Visualize time-specific Brier Score (calibration over time).
    """
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot BS(t) with confidence bands
    ax.errorbar(
        df_time['time_months'],
        df_time['bs_mean'],
        yerr=df_time['bs_sd'],
        marker='o',
        markersize=8,
        capsize=6,
        linewidth=2.5,
        color='darkorange',
        ecolor='bisque',
        label='Mean BS(t) ± SD'
    )

    ax.fill_between(
        df_time['time_months'],
        df_time['bs_min'],
        df_time['bs_max'],
        alpha=0.15,
        color='darkorange',
        label='Min-Max Range'
    )

    # Reference line at 0.25 (poor calibration threshold)
    ax.axhline(0.25, color='red', linestyle='--', linewidth=2, label='Poor Calibration (BS=0.25)')

    # Add shaded region for "good calibration" (BS < 0.15)
    ax.axhspan(0, 0.15, alpha=0.1, color='green', label='Good Calibration Zone (BS<0.15)')

    ax.set_xlabel('Follow-up Time (months)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Brier Score(t)', fontsize=13, fontweight='bold')
    ax.set_title(f'Time-Dependent Calibration - {outcome_name}', fontsize=15, fontweight='bold')
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(alpha=0.3, linestyle='--')

    # Add time labels on x-axis
    ax.set_xticks(df_time['time_months'])
    ax.set_xticklabels(df_time['time_label'], rotation=45, ha='right')

    # Invert y-axis note (lower BS = better)
    ax.text(
        0.98, 0.98,
        'Lower is better ↓',
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    )

    plt.tight_layout()
    plt.show()


# ==================== USAGE EXAMPLE ====================
if __name__ == "__main__":
    # Define time grid
    times_eval_grid = np.array([
        0.5,   # 2 weeks
        1,     # 1 month
        3,     # 3 months
        6,     # 6 months
        12,    # 1 year
        36,    # 3 years
        60,    # 5 years
        120    # 10 years
    ])

    # Run calibration analysis
    df_global_calib, df_time_calib, df_importance_calib = (
        permutation_importance_calibration_timespecific_cv_mi(
            X_list=X_reduced_list,
            y_surv_list=y_surv_readm_list_corrected,
            times_eval=times_eval_grid,
            alpha_idx=49,  # Use last alpha (least regularization)
            n_splits=5,
            n_repeats=20,
            random_state=2125,
            # Corrected parameters to avoid model collapse
            l1_ratio=0.1,              # More Ridge
            alpha_min_ratio=0.001,     # Allow smaller alphas
            n_alphas=50,
            max_iter=100000,
            n_jobs=-1
        )
    )

    # Display results
    print("\n" + "="*80)
    print("DETAILED GLOBAL CALIBRATION METRICS")
    print("="*80)
    print(df_global_calib.T)

    print("\n" + "="*80)
    print("DETAILED TIME-SPECIFIC CALIBRATION")
    print("="*80)
    print(df_time_calib)

    print("\n" + "="*80)
    print("TOP 20 FEATURES (CALIBRATION IMPORTANCE)")
    print("="*80)
    print(df_importance_calib.head(20))

    # Visualize
    plot_time_dependent_calibration(df_time_calib, outcome_name="Readmission")

For death

In [ ]:
df_death_calib, df_time_calib_death, df_importance_calib_death = (
    permutation_importance_calibration_timespecific_cv_mi(
        X_list=X_reduced_list,
        y_surv_list=y_surv_death_list,
        times_eval=times_eval_grid,
        alpha_idx=49,  # Use last alpha (least regularization)
        n_splits=10,
        n_repeats=20,
        random_state=2125,
        # Corrected parameters to avoid model collapse
        l1_ratio=0.1,              # More Ridge
        alpha_min_ratio=0.001,     # Allow smaller alphas
        n_alphas=50,
        max_iter=100000,
        n_jobs=-1
    )
)
# Display results
print("\n" + "="*80)
print("DETAILED GLOBAL CALIBRATION METRICS")
print("="*80)
print(df_death_calib.T)

print("\n" + "="*80)
print("DETAILED TIME-SPECIFIC CALIBRATION")
print("="*80)
print(df_time_calib_death)

print("\n" + "="*80)
print("TOP 20 FEATURES (CALIBRATION IMPORTANCE)")
print("="*80)
print(df_importance_calib_death.head(20))

# Visualize
plot_time_dependent_calibration(df_time_calib_death, outcome_name="Mortality")

## Pruebas

In [ ]:
# Verificar:
print(f"N total: {len(y_surv_death_list[0])}")
print(f"N eventos (muertes): {y_surv_death_list[0]['event'].sum()}")
print(f"Prevalencia: {y_surv_death_list[0]['event'].mean():.3f}")
print(f"Tiempos positivos: {(y_surv_death_list[0]['time'] > 0).all()}")

# Esperado para mortalidad:
# - Eventos: 50-200 (3-5% de ~4000)
# - Prevalencia: 0.03-0.05
# - Todos los tiempos > 0

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import (
    concordance_index_ipcw,
    cumulative_dynamic_auc,
    brier_score,
    integrated_brier_score
)
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import seaborn as sns

def coxph_unregularized_cv_mi(
    X_list,
    y_surv_list,
    times_eval,
    n_splits=5,
    random_state=2125,
    n_jobs=-1,
    verbose=True
):
    """
    CoxPH sin regularización con múltiples imputaciones y CV.

    Útil para:
    1. Verificar si los datos tienen señal predictiva
    2. Obtener coeficientes interpretables (sin shrinkage)
    3. Baseline de comparación con Coxnet

    Parameters
    ----------
    X_list : list of pandas.DataFrame
        Lista de datasets imputados
    y_surv_list : list of structured arrays
        Lista de outcomes Surv(event, time)
    times_eval : array-like
        Tiempos para evaluación
    n_splits : int
        Número de folds CV
    random_state : int
        Semilla para reproducibilidad
    n_jobs : int
        Trabajos paralelos
    verbose : bool
        Imprimir progreso

    Returns
    -------
    results : dict
        Diccionario con métricas y coeficientes
    df_metrics : pandas.DataFrame
        Métricas globales y por tiempo
    df_coefs : pandas.DataFrame
        Coeficientes agregados con estadísticas
    """
    # Preparación
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_list_np = [X.values.astype(float) for X in X_list]
    n_imputations = len(X_list)
    n = X_list_np[0].shape[0]

    # Filtrar tiempos
    max_time = np.max([y['time'].max() for y in y_surv_list])
    times_eval = np.array(times_eval)
    times_eval = times_eval[times_eval <= max_time]
    n_times = len(times_eval)

    # CV splits
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = list(kf.split(np.arange(n)))

    def format_time_label(months):
        if months < 1:
            return f"{int(months*4)}wk"
        elif months < 12:
            return f"{int(months)}mo"
        else:
            return f"{int(months/12)}yr"

    def compute_fold(d, fold_idx, train_idx, test_idx):
        """Compute metrics for one fold."""
        if verbose:
            print(f"  Imputation {d+1}/{n_imputations}, fold {fold_idx+1}/{n_splits}")

        X_imp = X_list_np[d]
        y_imp = y_surv_list[d]

        X_train = X_imp[train_idx]
        X_test = X_imp[test_idx]
        y_train = y_imp[train_idx]
        y_test = y_imp[test_idx]

        # Fit CoxPH sin regularización
        try:
            model = CoxPHSurvivalAnalysis(alpha=0.0, verbose=0)
            model.fit(X_train, y_train)

            # Predictions
            risk_scores = model.predict(X_test)

            # 1. C-index
            c_index = concordance_index_ipcw(y_train, y_test, risk_scores)[0]

            # 2. Time-specific AUC
            try:
                auc_scores, mean_auc = cumulative_dynamic_auc(
                    y_train, y_test, risk_scores, times=times_eval
                )
            except:
                auc_scores = np.full(n_times, np.nan)
                mean_auc = np.nan

            # 3. Time-specific Brier Score
            try:
                # Get survival functions
                surv_funcs = model.predict_survival_function(X_test)
                # Evaluate at times_eval
                S_test = np.array([[fn(t) for t in times_eval] for fn in surv_funcs])

                bs_scores = []
                for i, t in enumerate(times_eval):
                    try:
                        _, bs_t = brier_score(y_train, y_test, S_test, times=[t])
                        bs_scores.append(bs_t[0])
                    except:
                        bs_scores.append(np.nan)
                bs_scores = np.array(bs_scores)

                # IBS
                ibs = integrated_brier_score(y_train, y_test, S_test, times=times_eval)
            except Exception as e:
                if verbose:
                    print(f"    Warning: Brier score failed - {str(e)}")
                bs_scores = np.full(n_times, np.nan)
                ibs = np.nan

            # 4. Coeficientes
            coefs = model.coef_.copy()

            return {
                'success': True,
                'c_index': c_index,
                'auc_scores': auc_scores,
                'bs_scores': bs_scores,
                'ibs': ibs,
                'coefs': coefs,
                'n_coefs_nonzero': np.sum(coefs != 0),
                'max_abs_coef': np.max(np.abs(coefs))
            }

        except Exception as e:
            if verbose:
                print(f"    ERROR: {str(e)}")
            return {
                'success': False,
                'c_index': np.nan,
                'auc_scores': np.full(n_times, np.nan),
                'bs_scores': np.full(n_times, np.nan),
                'ibs': np.nan,
                'coefs': np.full(n_features, np.nan),
                'n_coefs_nonzero': 0,
                'max_abs_coef': 0.0
            }

    # ==================== PARALLEL EXECUTION ====================
    if verbose:
        print(f"\n{'='*80}")
        print(f"CoxPH UNREGULARIZED - MULTIPLE IMPUTATION + CV")
        print(f"{'='*80}")
        print(f"Imputations: {n_imputations}, CV folds: {n_splits}")
        print(f"Features: {n_features}, Samples: {n}")
        print(f"Time points: {n_times}")
        print(f"{'='*80}\n")

    results = Parallel(n_jobs=n_jobs, backend='loky', verbose=10 if verbose else 0)(
        delayed(compute_fold)(d, fold_idx, train_idx, test_idx)
        for d in range(n_imputations)
        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits)
    )

    # Filter successful results
    successful_results = [r for r in results if r['success']]
    n_successful = len(successful_results)

    if n_successful == 0:
        raise ValueError("All folds failed to converge. Check your data.")

    if verbose and n_successful < len(results):
        print(f"\n⚠️ Warning: {len(results) - n_successful}/{len(results)} folds failed")

    # ==================== AGGREGATE METRICS ====================

    # C-index
    c_indices = [r['c_index'] for r in successful_results]
    c_mean = np.mean(c_indices)
    c_sd = np.std(c_indices, ddof=1) if len(c_indices) > 1 else 0.0

    # AUC by time
    all_aucs = np.array([r['auc_scores'] for r in successful_results])
    auc_mean_global = np.nanmean(all_aucs)
    auc_sd_global = np.nanstd(all_aucs, ddof=1)

    # BS by time
    all_bs = np.array([r['bs_scores'] for r in successful_results])
    bs_mean_global = np.nanmean(all_bs)
    bs_sd_global = np.nanstd(all_bs, ddof=1)

    # IBS
    ibs_list = [r['ibs'] for r in successful_results if not np.isnan(r['ibs'])]
    ibs_mean = np.mean(ibs_list) if len(ibs_list) > 0 else np.nan
    ibs_sd = np.std(ibs_list, ddof=1) if len(ibs_list) > 1 else 0.0

    # Coeficientes
    all_coefs = np.array([r['coefs'] for r in successful_results])
    coef_mean = np.nanmean(all_coefs, axis=0)
    coef_sd = np.nanstd(all_coefs, axis=0, ddof=1)
    coef_min = np.nanmin(all_coefs, axis=0)
    coef_max = np.nanmax(all_coefs, axis=0)

    # Número de coeficientes no-cero
    n_nonzero = [r['n_coefs_nonzero'] for r in successful_results]
    max_abs_coefs = [r['max_abs_coef'] for r in successful_results]

    # ==================== TIME-SPECIFIC METRICS ====================
    time_rows = []
    for i, t in enumerate(times_eval):
        auc_at_t = all_aucs[:, i]
        bs_at_t = all_bs[:, i]

        time_rows.append({
            'time_months': t,
            'time_label': format_time_label(t),
            'auc_mean': np.nanmean(auc_at_t),
            'auc_sd': np.nanstd(auc_at_t, ddof=1),
            'bs_mean': np.nanmean(bs_at_t),
            'bs_sd': np.nanstd(bs_at_t, ddof=1),
            'n_evals': np.sum(~np.isnan(auc_at_t))
        })

    df_time = pd.DataFrame(time_rows)

    # ==================== COEFFICIENT TABLE ====================
    df_coefs = pd.DataFrame({
        'feature': feature_names,
        'coef_mean': coef_mean,
        'coef_sd': coef_sd,
        'coef_min': coef_min,
        'coef_max': coef_max,
        'abs_coef_mean': np.abs(coef_mean),
        'is_stable': (np.sign(coef_min) == np.sign(coef_max)) & (coef_mean != 0)
    })
    df_coefs = df_coefs.sort_values('abs_coef_mean', ascending=False).reset_index(drop=True)

    # ==================== GLOBAL METRICS TABLE ====================
    df_global = pd.DataFrame([{
        'c_index_mean': c_mean,
        'c_index_sd': c_sd,
        'auc_mean_global': auc_mean_global,
        'auc_sd_global': auc_sd_global,
        'bs_mean_global': bs_mean_global,
        'bs_sd_global': bs_sd_global,
        'ibs_mean': ibs_mean,
        'ibs_sd': ibs_sd,
        'n_successful_folds': n_successful,
        'n_total_folds': len(results),
        'n_features': n_features,
        'n_nonzero_coefs_mean': np.mean(n_nonzero),
        'max_abs_coef_mean': np.mean(max_abs_coefs)
    }])

    # ==================== COMBINE METRICS ====================
    df_metrics = pd.concat([
        df_global.T.rename(columns={0: 'value'}).assign(category='global'),
        df_time.set_index('time_label').stack().reset_index().rename(
            columns={'level_1': 'metric', 0: 'value'}
        ).assign(category='time_specific')
    ], ignore_index=True)

    # ==================== PRINT SUMMARY ====================
    if verbose:
        print(f"\n{'='*80}")
        print(f"RESULTS SUMMARY")
        print(f"{'='*80}")

        print(f"\n>>> GLOBAL PERFORMANCE <<<")
        print(f"C-index:       {c_mean:.4f} ± {c_sd:.4f}")
        print(f"Mean AUC(t):   {auc_mean_global:.4f} ± {auc_sd_global:.4f}")
        print(f"Mean BS(t):    {bs_mean_global:.4f} ± {bs_sd_global:.4f}")
        print(f"IBS:           {ibs_mean:.4f} ± {ibs_sd:.4f}")

        print(f"\n>>> MODEL COMPLEXITY <<<")
        print(f"Features:              {n_features}")
        print(f"Avg non-zero coefs:    {np.mean(n_nonzero):.1f}")
        print(f"Max |coef| (avg):      {np.mean(max_abs_coefs):.4f}")

        print(f"\n>>> CONVERGENCE <<<")
        print(f"Successful folds:      {n_successful}/{len(results)}")

        print(f"\n>>> TIME-SPECIFIC PERFORMANCE <<<")
        print(df_time[['time_label', 'auc_mean', 'auc_sd', 'bs_mean', 'bs_sd']].to_string(index=False))

        print(f"\n>>> TOP 15 COEFFICIENTS (by absolute value) <<<")
        print(df_coefs.head(15)[['feature', 'coef_mean', 'coef_sd', 'is_stable']].to_string(index=False))

        print(f"\n{'='*80}")

        # Diagnostic messages
        if c_mean < 0.52:
            print(f"⚠️ WARNING: C-index very low ({c_mean:.4f})")
            print("   → Model is not discriminating well")
            print("   → Check: outcome coding, feature quality, sample size")
        elif c_mean < 0.60:
            print(f"⚠️ C-index marginal ({c_mean:.4f})")
            print("   → Weak signal - consider feature engineering")
        else:
            print(f"✅ C-index acceptable ({c_mean:.4f})")
            print("   → Model has discriminative power")

        if np.mean(max_abs_coefs) < 0.01:
            print(f"\n⚠️ WARNING: Max coef very small ({np.mean(max_abs_coefs):.6f})")
            print("   → Features may need scaling or have weak effects")

        print(f"{'='*80}\n")

    # ==================== RETURN RESULTS ====================
    results_dict = {
        'c_indices': c_indices,
        'auc_scores_all': all_aucs,
        'bs_scores_all': all_bs,
        'ibs_list': ibs_list,
        'coefs_all': all_coefs,
        'times_eval': times_eval,
        'successful_folds': n_successful,
        'total_folds': len(results)
    }

    return results_dict, df_metrics, df_coefs, df_time


# ==================== VISUALIZATION FUNCTIONS ====================

def plot_coxph_coefficients(df_coefs, top_n=20, figsize=(12, 8)):
    """
    Visualizar coeficientes del CoxPH con intervalos de confianza.
    """
    df_plot = df_coefs.head(top_n).copy()

    fig, ax = plt.subplots(figsize=figsize)

    # Color por signo
    colors = ['red' if c < 0 else 'steelblue' for c in df_plot['coef_mean']]

    # Bar plot con error bars
    y_pos = np.arange(len(df_plot))
    ax.barh(y_pos, df_plot['coef_mean'], xerr=df_plot['coef_sd'],
            color=colors, alpha=0.7, capsize=5)

    # Añadir línea en x=0
    ax.axvline(0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)

    # Labels
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_plot['feature'], fontsize=10)
    ax.set_xlabel('Coefficient (Mean ± SD)', fontsize=12, fontweight='bold')
    ax.set_title(f'Top {top_n} CoxPH Coefficients (Unregularized)',
                 fontsize=14, fontweight='bold')

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='steelblue', alpha=0.7, label='Hazard ↑ (positive)'),
        Patch(facecolor='red', alpha=0.7, label='Hazard ↓ (negative)')
    ]
    ax.legend(handles=legend_elements, loc='lower right')

    # Grid
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()

    plt.tight_layout()
    plt.show()


def plot_coefficient_stability(df_coefs, top_n=20, figsize=(14, 8)):
    """
    Visualizar estabilidad de coeficientes (min/max range).
    """
    df_plot = df_coefs.head(top_n).copy()

    fig, ax = plt.subplots(figsize=figsize)

    y_pos = np.arange(len(df_plot))

    # Plot mean with range
    for i, row in df_plot.iterrows():
        # Range bar
        ax.plot([row['coef_min'], row['coef_max']], [i, i],
                color='lightgray', linewidth=6, alpha=0.5, zorder=1)

        # Mean point
        color = 'green' if row['is_stable'] else 'orange'
        marker = 'o' if row['is_stable'] else 'x'
        ax.scatter(row['coef_mean'], i, color=color, s=100, marker=marker,
                  zorder=3, edgecolor='black', linewidth=1.5)

    # Zero line
    ax.axvline(0, color='red', linestyle='--', linewidth=2, alpha=0.7)

    # Labels
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_plot['feature'], fontsize=10)
    ax.set_xlabel('Coefficient Value', fontsize=12, fontweight='bold')
    ax.set_title(f'Coefficient Stability Across CV Folds (Top {top_n})',
                 fontsize=14, fontweight='bold')

    # Legend
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='green',
               markersize=10, markeredgecolor='black', label='Stable (same sign)'),
        Line2D([0], [0], marker='x', color='w', markerfacecolor='orange',
               markersize=10, markeredgewidth=2, label='Unstable (sign flips)'),
        Patch(facecolor='lightgray', alpha=0.5, label='Min-Max range')
    ]
    ax.legend(handles=legend_elements, loc='lower right')

    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()

    plt.tight_layout()
    plt.show()


def compare_coxph_vs_coxnet(df_coefs_coxph, df_coefs_coxnet, top_n=15):
    """
    Comparar coeficientes CoxPH vs Coxnet lado a lado.
    """
    # Merge por feature
    df_compare = pd.merge(
        df_coefs_coxph[['feature', 'coef_mean']].rename(columns={'coef_mean': 'coxph'}),
        df_coefs_coxnet[['feature', 'coef_mean']].rename(columns={'coef_mean': 'coxnet'}),
        on='feature',
        how='outer'
    ).fillna(0)

    # Ordenar por CoxPH
    df_compare['abs_coxph'] = np.abs(df_compare['coxph'])
    df_compare = df_compare.sort_values('abs_coxph', ascending=False).head(top_n)

    # Plot
    fig, ax = plt.subplots(figsize=(14, 8))

    x = np.arange(len(df_compare))
    width = 0.35

    ax.bar(x - width/2, df_compare['coxph'], width, label='CoxPH (unregularized)',
           color='steelblue', alpha=0.8)
    ax.bar(x + width/2, df_compare['coxnet'], width, label='Coxnet (regularized)',
           color='darkorange', alpha=0.8)

    ax.set_ylabel('Coefficient', fontsize=12, fontweight='bold')
    ax.set_title('Coefficient Comparison: CoxPH vs Coxnet', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df_compare['feature'], rotation=45, ha='right', fontsize=9)
    ax.legend()
    ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

    return df_compare


# ==================== USAGE EXAMPLE ====================
if __name__ == "__main__":

    # Define time grid
    times_eval_grid = np.array([0.5, 1, 3, 6, 12, 36, 60, 120])

    print("\n" + "🔬"*40)
    print("DIAGNOSTIC TEST: CoxPH WITHOUT REGULARIZATION")
    print("🔬"*40)

    # ==================== READMISSION ====================
    print("\n>>> OUTCOME: READMISSION <<<\n")

    results_coxph_readm, df_metrics_readm, df_coefs_readm, df_time_readm = (
        coxph_unregularized_cv_mi(
            X_list=X_reduced_list,
            y_surv_list=y_surv_readm_list_corrected,
            times_eval=times_eval_grid,
            n_splits=5,
            random_state=2125,
            n_jobs=-1,
            verbose=True
        )
    )

    # Visualize coefficients
    plot_coxph_coefficients(df_coefs_readm, top_n=20)
    plot_coefficient_stability(df_coefs_readm, top_n=20)

    # ==================== MORTALITY ====================
    print("\n>>> OUTCOME: MORTALITY <<<\n")

    results_coxph_death, df_metrics_death, df_coefs_death, df_time_death = (
        coxph_unregularized_cv_mi(
            X_list=X_reduced_list,
            y_surv_list=y_surv_death_list,  # ← CAMBIAR al outcome correcto
            times_eval=times_eval_grid,
            n_splits=5,
            random_state=2125,
            n_jobs=-1,
            verbose=True
        )
    )

    # Visualize coefficients
    plot_coxph_coefficients(df_coefs_death, top_n=20)
    plot_coefficient_stability(df_coefs_death, top_n=20)

    # ==================== SAVE RESULTS ====================
    df_coefs_readm.to_csv('coxph_coefs_readmission.csv', index=False)
    df_coefs_death.to_csv('coxph_coefs_mortality.csv', index=False)

    print("\n✅ CoxPH analysis complete. Coefficients saved.")

    # ==================== INTERPRETATION GUIDE ====================
    print("\n" + "="*80)
    print("INTERPRETATION GUIDE")
    print("="*80)
    print("""
    C-index Interpretation:
    - < 0.52  → No discriminative power (baseline model)
    - 0.52-0.60 → Weak discrimination (marginal signal)
    - 0.60-0.70 → Moderate discrimination (acceptable)
    - > 0.70  → Good discrimination (strong signal)

    Coefficient Interpretation:
    - Positive coef → ↑ Hazard (worse prognosis)
    - Negative coef → ↓ Hazard (better prognosis)
    - HR = exp(coef) → Hazard Ratio per 1-unit increase

    Stability:
    - is_stable = True → Coefficient sign consistent across folds
    - is_stable = False → Unstable effect (multicollinearity?)

    Next Steps:
    1. If C-index > 0.55 → Data has signal, use Coxnet with less regularization
    2. If C-index ≈ 0.50 → Problem with data/outcome, investigate further
    3. Compare coefs with Coxnet to see shrinkage effect
    """)
    print("="*80)

### Compare to KM

#### Readmission

In [ ]:
from sksurv.metrics import integrated_brier_score
from sksurv.linear_model import CoxPHSurvivalAnalysis # Or just use KM directly
from sksurv.nonparametric import kaplan_meier_estimator
import numpy as np

# 1. Calculate the Kaplan-Meier Estimate (The "Null" Prediction)
# We need to create a probability matrix where every row is the KM curve
km_times, km_probs = kaplan_meier_estimator(y_surv_readm["event"], y_surv_readm["time"])

# Interpolate KM probabilities to match your 'times_eval'
# This creates a "Null Prediction" where every patient gets the exact same probability
null_probs = np.interp(times_eval_readm, km_times, km_probs)
null_preds = np.tile(null_probs, (len(y_surv_readm), 1))

# 2. Calculate IBS for the Null Model
null_ibs = integrated_brier_score(y_surv_readm, y_surv_readm, null_preds, times_eval_readm)

print(f"Model IBS: 0.1427")
print(f"Null (KM) IBS: {null_ibs:.4f}")

if 0.1427 < null_ibs - 0.005:
    print("VERDICT: The model has predictive power.")
else:
    print("VERDICT: The model is NOT performing better than random guessing (Kaplan-Meier).")

#### Death

In [ ]:
from sksurv.metrics import integrated_brier_score
from sksurv.nonparametric import kaplan_meier_estimator
import numpy as np

# 1. Calculate the Kaplan-Meier Estimate (The "Average" Prediction)
km_times, km_probs = kaplan_meier_estimator(y_surv_death["event"], y_surv_death["time"])

# 2. Interpolate KM probabilities to match your evaluation times
# Note: Ensure 'times_eval_death' is defined in your environment
null_probs = np.interp(times_eval_death, km_times, km_probs)

# 3. Create the "Null Prediction" matrix (Same prediction for everyone)
null_preds = np.tile(null_probs, (len(y_surv_death), 1))

# 4. Calculate IBS for the Null Model
null_ibs_death = integrated_brier_score(y_surv_death, y_surv_death, null_preds, times_eval_death)

print(f"Model IBS:  0.0351") # From your logs
print(f"Null IBS:   {null_ibs_death:.4f}")

diff = 0.0351 - null_ibs_death

if abs(diff) < 0.001:
    print("\nCONCLUSION: The model is effectively ignoring all predictors.")
    print("It provides no value over a simple population average.")
else:
    print(f"\nCONCLUSION: There is a tiny difference ({diff:.5f}), but likely not clinically relevant.")

In [ ]:
#summary = df.groupby('group').describe()